In [ ]:
import json
import os
from pathlib import Path
from typing import Any

import pandas as pd

from kibad_llm.config import PROJ_ROOT

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)
# set wider column width for displaying pandas data frames
pd.set_option("max_colwidth", 400)

In [ ]:
METRICS_DIR = "data/prediction_results/159_core_schema_baseline_multirun/logs/evaluate"
ERRORS_DIR = "data/prediction_results/159_core_schema_baseline_multirun/logs/evaluate_errors"

In [ ]:
# load evaluation job_return_value.json files
def _load_evaluations(parent_dir: Path) -> dict:
    log_filename = "job_return_value.json"
    # get sub directories, 1 level only
    run_dirs = [p for p in Path(parent_dir).iterdir() if p.is_dir()]
    # assume that each subdir contains a 'job_return_value.json' from a multi-run evaluation
    # (i.e. it contains a parent key indictating the run, may be empty, and sub-keys 'prediction' and 'overrides' that store metadata)
    data = [json.loads((subdir / log_filename).read_text()) for subdir in run_dirs]
    # keep the keys / identifiers?
    # runs = {key: subdict for d in data for key, subdict in d.items()}
    runs = [subdict for d in data for subdict in d.values()]
    return runs


eval_runs = _load_evaluations(METRICS_DIR)

In [ ]:
# simplify the dictionaries and add group keys


def _flatten_dict(d: dict[str, Any], sep: str = ".") -> dict[str, Any]:
    result: dict[str, Any] = dict()
    for k, v in d.items():
        if isinstance(v, (str, int, float, bool)):
            result[k] = v
        elif isinstance(v, dict):
            for e in v:
                result[f"{k}{sep}{e}"] = v[e]
    return result


def _clean_metadata(run_dict: dict) -> (dict, dict):
    metadata_keys = ["prediction", "overrides"]
    metadata = {}
    for k in metadata_keys:
        metadata[k] = run_dict.pop(k)

    return _flatten_dict(run_dict), _flatten_dict(metadata)


metrics, metadata = map(list, zip(*[_clean_metadata(run) for run in eval_runs]))

In [ ]:
metrics_df = pd.DataFrame.from_records(metrics)

In [ ]:
# add some metadata keys
metrics_df["overrides.pdf_directory"] = [str(m["overrides.pdf_directory"]) for m in metadata]
metrics_df["overrides.extractor/llm"] = [
    str(m.get("overrides.extractor/llm", "gpt_oss_20b")) for m in metadata
]  # may not be set for default gpt_oss_20b
metrics_df["overrides.experiment/predict"] = [
    str(m["overrides.experiment/predict"]) for m in metadata
]
# metrics_df["overrides.request_parameters.extra_body.seed"] = [str(m["overrides.request_parameters.extra_body.seed"]) for m in metadata]

In [ ]:
metrics_df_avg = metrics_df.groupby("overrides.extractor/llm", as_index=False).agg(
    {col: "first" if metrics_df[col].dtype == "object" else "mean" for col in metrics_df.columns}
)
metrics_df_avg

In [ ]:
cols_to_plot = [col for col in metrics_df_avg.columns if col == "ALL.f1"]
metrics_df_avg.plot(kind="bar", x="overrides.extractor/llm", y=cols_to_plot)

In [ ]:
cols_to_plot = [
    col for col in metrics_df_avg.columns if col.endswith("f1") and not col.startswith("AVG")
]
metrics_df_avg.plot(kind="bar", x="overrides.extractor/llm", y=cols_to_plot)

In [ ]:
error_runs = _load_evaluations(ERRORS_DIR)
errors, errors_metadata = map(list, zip(*[_clean_metadata(run) for run in error_runs]))

errors_df = pd.DataFrame.from_records(errors)
# add some metadata keys
errors_df["overrides.pdf_directory"] = [str(m["overrides.pdf_directory"]) for m in errors_metadata]
errors_df["overrides.extractor/llm"] = [
    str(m.get("overrides.extractor/llm", "gpt_oss_20b")) for m in errors_metadata
]  # may not be set for default gpt_oss_20b
errors_df["overrides.experiment/predict"] = [
    str(m["overrides.experiment/predict"]) for m in errors_metadata
]
errors_df_avg = errors_df.groupby("overrides.extractor/llm", as_index=False).agg(
    {col: "first" if errors_df[col].dtype == "object" else "mean" for col in errors_df.columns}
)
errors_df_avg

In [ ]:
cols_to_plot = errors_df_avg.select_dtypes(include="number").columns
errors_df_avg.plot(kind="bar", x="overrides.extractor/llm", y=cols_to_plot)